## **Part I: Market Overview**

This section looks at the overall genre landscape and the genres that reached the top 3 in market share across the historical (2000–2020) and recent (2021–2025) periods.

To keep the analysis readable, I focused on the five genres with the highest average market share in each summary table. Part II will expand beyond the top five to look for broader market trends and changes.

In [1]:
import pandas as pd
import sys
sys.path.append("../scripts")
import utilities 
import sqlite3

In [2]:
spotify_2025 = pd.read_csv('../data/spotify_data_clean.csv')
spotify_prev = pd.read_csv('../data/track_data_final.csv')

### **Data Cleaning**

To have the cleaning process consistent and resuable, utility functions were created to:
* Standardize track duration to seconds
* Convert numeric columns to the appropiate data types
* Standardize album release dates and extract release years
* Clean and split artist genre values based on each dataset's formatting

In [3]:
# applying cleaning functions from utilities
spotify_2025 = utilities.clean_data(spotify_2025)
spotify_2025 = utilities.clean_comma_separated(spotify_2025, "artist_genres")

spotify_prev = utilities.clean_data(spotify_prev)
spotify_prev = utilities.clean_stringified_list(spotify_prev, "artist_genres")

In [4]:
# preserving copies of original data for later analysis
# 'spotify_prev_all and spotify_2025_all as original dfs
# 'spotify_prev and spotify_2025 as working dfs for analysis
spotify_prev_all = spotify_prev.copy()
spotify_2025_all = spotify_2025.copy()

### Preparing Genres for Analysis

After cleaning the data, I exploded 'artist_genres' so each genre could be anaylzed separately. Since a track can belong to multiple genres, I assigned equal weights to prevent those tracks from being overcounted.
> *For example, a track associated with 2 genres controbutes 0.5 to each genre, keeping the total weight of each track equal to 1*
So, weighted track shows how much one individual track contributes to a genre

'release_year' was extracted from the album release data to analyze genre performance at a yearly level overtime

In [5]:
# exploding arist_genre column
spotify_2025 = spotify_2025.explode("artist_genres")
spotify_prev = spotify_prev.explode("artist_genres")

In [6]:
spotify_prev.head()

,track_id,track_name,track_number,track_popularity,track_duration_s,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_type,album_total_tracks
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61.0,213.173,False,Britney Spears,80.0,17755451,pop,325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,compilation,58
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67.0,158.760,False,BUNT.,69.0,293734,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,single,1
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63.0,225.328,False,Taylor Swift,100.0,145396321,<NA>,4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,album,26
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72.0,158.463,True,Billie Eilish,90.0,118692183,<NA>,0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,album,16
4,0GLfodYacy3BJE7AI3A8en,Man Down,7,57.0,267.013,False,Rihanna,90.0,68997177,<NA>,5QG3tjE5L9F6O2vCAPph38,Loud,2010-01-01,album,13


In [7]:
spotify_prev["artist_genres"].value_counts(dropna=False)

artist_genres
<NA>             4492
soundtrack        451
pop               386
rap               304
country           276
                 ... 
electro swing       1
uk garage           1
space music         1
desi pop            1
classic blues       1
Name: count, Length: 425, dtype: int64

In [8]:
spotify_2025["artist_genres"].value_counts(dropna=False)

artist_genres
<NA>                  3361
pop                    920
country                680
soundtrack             450
hip hop                431
                      ... 
shatta                   1
spoken word              1
dembow                   1
indonesian hip hop       1
classic blues            1
Name: count, Length: 431, dtype: int64

In [9]:
# dropping null values before analysis
spotify_prev = spotify_prev.dropna(subset=['artist_genres'])
spotify_2025 = spotify_2025.dropna(subset=['artist_genres'])

In [10]:
# assigning weights to genres
spotify_prev['genre_weight'] = 1/spotify_prev.groupby('track_id')['artist_genres'].transform('count')
spotify_2025['genre_weight'] = 1/spotify_2025.groupby('track_id')['artist_genres'].transform('count')

In [11]:
# extracting year from album_release_date
spotify_prev['release_year'] = spotify_prev['album_release_date'].dt.year.astype('Int64')
spotify_2025['release_year'] = spotify_2025['album_release_date'].dt.year.astype('Int64')

In [ ]:
# checking if 'genre_weight' & 'release_year' have been added
spotify_prev.head()

,track_id,track_name,track_number,track_popularity,track_duration_s,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_type,album_total_tracks,genre_weight,release_year
0,6pymOcrCnMuCWdgGVTvUgP,3,57,61.0,213.173,False,Britney Spears,80.0,17755451,pop,325wcm5wMnlfjmKZ8PXIIn,The Singles Collection,2009-11-09,compilation,58,1.00,2009
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67.0,158.760,False,BUNT.,69.0,293734,stutter house,2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,single,1,1.00,2023
5,7H0ya83CMmgFcOhw0UB6ow,Space Song,3,77.0,320.466,False,Beach House,72.0,2803036,dream pop,194CqC2Zi0kUFEPWedb3qr,Depression Cherry,2015-08-28,album,9,1.00,2015
7,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52.0,145.977,False,Alok,76.0,11247155,brazilian bass,1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,single,2,0.25,2024
7,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52.0,145.977,False,Alok,76.0,11247155,electronic,1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,single,2,0.25,2024


### Measuring Genre Dominance

I used SQL to create the main genre level metrics for each year:
* **Weighted Tracks**: Total genre's contribution within a year 
  * (*sum of genre weights*) 
* **Market Share**: A genre's share of total weighted tracks within each year 
  * (*genre's weighted tracks/total weighted tracks in a year*)
* **Average Popularity**: Average track popularity for each genre by year
* **Genre Rank**: Each genre's yearly rank based on highest market share

The results were loaded back into Pandas for further filtering and analysis

In [ ]:
# creating sqlite database
conn = sqlite3.connect('../spotify.db')

# loading datasets into sqlite as a table
spotify_2025.to_sql("spotify_2025", conn, if_exists = "replace", index=False) 
spotify_prev.to_sql("spotify_prev", conn, if_exists = "replace", index=False) 

In [ ]:
# opens & reads genre_dominance sql script, then closes file
with open('../sql/genre_dominance.sql') as f:
    query = f.read()

# executing the query and storing results into 'market_share_df'
market_share_df = pd.read_sql_query(query, conn)

In [15]:
market_share_df.head(20)

,artist_genres,release_year,weighted_tracks,market_share,avg_popularity,genre_rank
0,christian country,1952,0.166667,0.166667,45.000000,1
1,classic country,1952,0.166667,0.166667,45.000000,1
2,country,1952,0.166667,0.166667,45.000000,1
3,honky tonk,1952,0.166667,0.166667,45.000000,1
4,outlaw country,1952,0.166667,0.166667,45.000000,1
5,traditional country,1952,0.166667,0.166667,45.000000,1
6,blues,1958,0.250000,0.250000,55.000000,1
7,classic blues,1958,0.250000,0.250000,55.000000,1
8,rock and roll,1958,0.250000,0.250000,55.000000,1
9,rockabilly,1958,0.250000,0.250000,55.000000,1


### Filtering Sparse Years

Some earlier years contained very few tracks, which could make their market shares and rankings less representative. To reduce the impact of sparse data, I created a minimum yearly sample threshold using the 25th percentile of yearly track counts, with a minimum floor of 25 tracks.

In [ ]:
# checking for sparse data, specifically in earlier years
tracks_per_year = spotify_prev.groupby('release_year')['track_id'].nunique().sort_index()

In [17]:
tracks_per_year.head()

release_year
1952    1
1958    1
1959    5
1960    1
1962    3
Name: track_id, dtype: int64

In [ ]:
# checking the quantiles
tracks_per_year.describe()

count     68.000000
mean      63.029412
std       81.922054
min        1.000000
25%        5.750000
50%       24.000000
75%       93.750000
max      319.000000
Name: track_id, dtype: float64

In [ ]:
# filtering sparse data since early years with sparse data aren't really representative with what's relevant today
year_threshold = tracks_per_year.quantile(0.25) # value is 5.75 (6 or fewer tracks)
valid_years = tracks_per_year[tracks_per_year >= year_threshold].index

In [20]:
# applying filter to market_share_df
market_share_filtered = market_share_df[market_share_df['release_year'].isin(valid_years)]
market_share_filtered.head(20)

,artist_genres,release_year,weighted_tracks,market_share,avg_popularity,genre_rank
37,soundtrack,1966,2.000000,0.285714,26.0,1
38,classic rock,1966,1.000000,0.142857,68.5,2
39,classic soul,1966,1.000000,0.142857,59.0,2
40,baroque pop,1966,0.500000,0.071429,1.0,4
41,folk rock,1966,0.500000,0.071429,43.0,4
42,proto-punk,1966,0.500000,0.071429,1.0,4
43,psychedelic rock,1966,0.500000,0.071429,55.0,4
44,rock,1966,0.500000,0.071429,82.0,4
45,singer-songwriter,1966,0.500000,0.071429,43.0,4
46,soundtrack,1967,5.000000,0.555556,49.2,1


In [21]:
market_share_filtered['artist_genres'].value_counts()

artist_genres
rock               41
country            39
soundtrack         36
metal              31
classic country    30
                   ..
latin jazz          1
melodic house       1
dansk rap           1
djent               1
afro adura          1
Name: count, Length: 419, dtype: int64

In [ ]:
# reranking after filtering
market_share_filtered['genre_rank'] = (market_share_filtered.groupby('release_year')['market_share'].rank(method= 'min', ascending = False).astype(int))

In [23]:
# validating all genre_ranks begin at 1
market_share_filtered.groupby("release_year")["genre_rank"].min().value_counts()

genre_rank
1    51
Name: count, dtype: int64

These metrics were calculated in pandas instead of sql for simplicity reasons:
* **Hit Share**: A genre's proportion of total weighted hits across all genres within the year
  * (weighted hits/total weighted hits)
    * **Weighted Hits**: The contribution of an individual hit track to a genres, ajusted for multi-genre tracks
    * **Total Weighted Hits**: The sum of weighted hits for a specific genre within the given year

In [ ]:
# hit_share setup
# using the 75th percentile as the threshold to classify "hits"
hit_threshold = spotify_prev['track_popularity'].quantile(0.75)

# tracks "hits" as 1 and non-hits as 0
spotify_prev['is_hit'] = (spotify_prev['track_popularity'] >= hit_threshold).astype(int)

# weighted_hits adjusted for multi-genre tracks
spotify_prev['weighted_hits'] = (spotify_prev['is_hit'] * spotify_prev['genre_weight'])
hit_share_df = (spotify_prev.groupby(['release_year', 'artist_genres']).agg(weighted_hits = ('weighted_hits', 'sum')).reset_index())

In [25]:
spotify_prev['is_hit'].value_counts()

is_hit
0    5656
1    2004
Name: count, dtype: int64

In [26]:
# calculating hit_share
hit_share_df['total_weighted_hits'] = (hit_share_df.groupby('release_year')['weighted_hits'].transform('sum'))
hit_share_df['hit_share'] = (hit_share_df['weighted_hits']/hit_share_df['total_weighted_hits'])

In [27]:
# merging metrics together
market_share_filtered = market_share_filtered.merge(hit_share_df[['release_year', 'artist_genres', 'hit_share']], on = ['release_year', 'artist_genres'], how = 'left')
market_share_filtered.tail()

,artist_genres,release_year,weighted_tracks,market_share,avg_popularity,genre_rank,hit_share
2217,industrial rock,2025,0.200000,0.000704,55.0,115,0.0
2218,metal,2025,0.200000,0.000704,55.0,115,0.0
2219,nu metal,2025,0.200000,0.000704,55.0,115,0.0
2220,afro adura,2025,0.166667,0.000587,33.0,120,0.0
2221,afroswing,2025,0.166667,0.000587,33.0,120,0.0


In [ ]:
# popularity distributions for each genre-year combination
# contains median and standard deviation of popularity on top of already calculated metrics
popularity_dist_df = (spotify_prev.groupby(['release_year', 'artist_genres'])['track_popularity'].agg(median_popularity = 'median', std_popularity = 'std').reset_index())

In [29]:
# merging metrics together
market_share_filtered = market_share_filtered.merge(popularity_dist_df, on = ['release_year', 'artist_genres'], how = 'left')

In [48]:
market_share_filtered.head()

,artist_genres,release_year,weighted_tracks,market_share,avg_popularity,genre_rank,hit_share,median_popularity,std_popularity
0,soundtrack,1966,2.0,0.285714,26.0,1,0.0,26.0,36.769553
1,classic rock,1966,1.0,0.142857,68.5,2,0.5,68.5,19.091883
2,classic soul,1966,1.0,0.142857,59.0,2,0.0,59.0,NaN
3,baroque pop,1966,0.5,0.071429,1.0,4,0.0,1.0,NaN
4,folk rock,1966,0.5,0.071429,43.0,4,0.0,43.0,NaN


## **Part I Analysis**

### Recent Genre Performance (2021-2025)

To get a better idea of what the most recent music market looks like, I focused on genre performance from 2021-2025 (*which were the most up to date years in the dataset*). The metrics I used were market share, track volume, hit share, popularity, and consistency to see which genres had the strong presence and how well their track performed relative to their presence.

In [31]:
# filtering for 2021 - 2025 (last 5 years)
last_5_years = market_share_filtered[market_share_filtered['release_year'].between(2021, 2025)]

In [32]:
# overall summary of last 5 years
last_5_years_summary = (last_5_years
    .groupby('artist_genres')
    .agg(
        weighted_tracks = ('weighted_tracks', 'sum'),
        avg_market_share = ('market_share', 'mean'),
        avg_hit_share = ('hit_share', 'mean'),
        avg_popularity = ('avg_popularity', 'mean'),
        median_popularity = ('median_popularity', 'mean'),
        avg_std_popularity = ('std_popularity', 'mean'))
    .sort_values(by='avg_market_share', ascending = False)
)

In [33]:
last_5_years_summary

,weighted_tracks,avg_market_share,avg_hit_share,avg_popularity,median_popularity,avg_std_popularity
artist_genres,,,,,,
dark r&b,100.500000,0.081422,0.017671,48.536805,53.5,19.890618
pop,88.500000,0.064913,0.129807,64.470318,65.8,19.030354
soft pop,57.500000,0.059474,0.059408,55.373407,58.8,20.484921
soundtrack,46.500000,0.047104,0.002985,40.329464,42.4,15.271762
rap,49.333333,0.043244,0.097806,72.607106,73.7,7.491848
...,...,...,...,...,...,...
ranchera,0.100000,0.000426,0.001493,75.000000,75.0,NaN
cumbia norteña,0.100000,0.000426,0.001493,75.000000,75.0,NaN
grupera,0.100000,0.000426,0.001493,75.000000,75.0,NaN


> Dark R&B has the largest genre size at 100.5 weighted tracks and the highest average market share at 8.1%. However, its hit share is only 1.8% and its average popularity is relatively low at 48.5. Its high standard deviation of 19.9 also shows that popularity varies quite a bit between tracks. This suggests that Dark R&B's large market presence comes more from its track volume than from consistently popular or successful tracks.

> Pop has the second largest genre size at 88.5 weighted tracks and the second-highest average market share at 6.5%. Unlike Dark R&B, its market presence is backed by stronger track performance, with the highest hit share at 13% and a relatively high average popularity of 64.5. Its standard deviation of 19 still shows a fair amount of variation between tracks, but Pop generally combines a large market presence with strong performance.

> Soft Pop has 57.5 weighted tracks and an average market share of 6%, giving it a fairly strong presence in the recent market. Its 5.9% hit share and average popularity of 55.4 are more moderate, while its standard deviation of 20.5 is the highest among the top five. Soft Pop therefore has a solid market presence, but its individual tracks tend to perform less consistently.

> Soundtrack has 46.5 weighted tracks and an average market share of 4.7%, giving it a noticeable presence in the market. However, its hit share is only 0.3% and its average popularity of 40.3 is the lowest among the top five. Its standard deviation of 15.3 shows some variation between tracks. Overall, Soundtrack has a decent market presence, but its tracks tend to be less popular and rarely make up a large share of hits.

> Rap has a smaller market presence, with 49.3 weighted tracks and a 4.3% average market share. Despite this, it has the second highest hit share at 9.8% and the highest average popularity at 72.6. It also has the lowest standard deviation at 7.5, meaning its tracks tend to perform more consistently. Rap stands out as a smaller but high-performing genre, showing that strong track performance doesn't necessarily require a large market share.

**Summary**: *Overall, the last five years show that having a large market presence doesn't always mean stronger track performance. Dark R&B leads in market share and track volume but performs weaker in popularity and hit share, while Pop combines a large presence with stronger track performance. Rap stands out in the opposite way, achieving high popularity, hit share, and consistency despite its smaller market share. Soft Pop falls more in the middle but has less consistent track performance, while Soundtrack maintains a noticeable presence despite weaker popularity and hit performance.*

### Recent Market Leaders

Next, I looked specifically at the genres that ranked in the top 3 by market share from 2021-2025 (*using last_5_years*). I counted how often each genre reached the top 3 to see which genres were able to consistently hold a leading position in the market.

In [34]:
# top 3 genres in last 5 years (2021 - 2025)
recent_top_3 = last_5_years[last_5_years['genre_rank'] <= 3]

In [35]:
# summary of top 3 genres in last 5 years 
recent_top_3_summary = (recent_top_3
    .groupby('artist_genres')
    .agg(
        weighted_tracks = ('weighted_tracks', 'sum'),
        avg_market_share = ('market_share', 'mean'),
        avg_hit_share = ('hit_share', 'mean'),
        avg_popularity = ('avg_popularity', 'mean'),
        median_popularity = ('median_popularity', 'mean'),
        avg_std_popularity = ('std_popularity', 'mean'))
    .sort_values(by='avg_market_share', ascending = False)
)

In [36]:
# recent top 3 genre appearances (last 5 years)
recent_top_3[['artist_genres']].value_counts()

artist_genres
dark r&b         4
pop              2
soft pop         2
soundtrack       2
bedroom pop      1
hypertechno      1
k-pop            1
latin            1
rap              1
Name: count, dtype: int64

In [37]:
# adding appearance counts to the summary for more context of type of market lead
recent_top_3_summary['top_3_appearances'] = recent_top_3.groupby('artist_genres').size()

In [38]:
recent_top_3_summary

,weighted_tracks,avg_market_share,avg_hit_share,avg_popularity,median_popularity,avg_std_popularity,top_3_appearances
artist_genres,,,,,,,
soft pop,46.000000,0.124285,0.107463,43.719231,45.50,19.559360,2
pop,73.000000,0.120795,0.259592,70.892461,72.25,14.276937,2
k-pop,29.000000,0.102113,0.231707,77.310345,75.00,11.979457,1
soundtrack,30.500000,0.086779,0.000000,32.861538,35.75,14.275618,2
dark r&b,91.000000,0.085398,0.022089,49.546006,54.00,18.718887,4
bedroom pop,12.000000,0.082759,0.033333,57.916667,63.00,19.759731,1
hypertechno,25.500000,0.079937,0.000000,32.724138,33.00,12.363242,1
rap,11.833333,0.050355,0.141791,73.230769,75.00,9.065997,1
latin,8.833333,0.045068,0.122222,76.000000,76.00,4.320494,1


> Soft Pop had the highest average market share at 12.5%, but only appeared in the top 3 twice. Its track performance was also relatively weaker, with 10.7% hit share and an average popularity of 43.7. Its high standard deviation of 19.6 shows that popularity varied quite a bit across tracks. Overall, Soft Pop reached a strong market position when it made the top 3, but this leadership was occasional and wasn't matched by equally strong or consistent track performance.

> Pop appeared in the top 3 twice, with an average market share of 12.1%. During those appearances, it had the highest hit share at 26.0% and a high average popularity of 70.9, showing that its market position was supported by strong track performance. Its standard deviation of 14.3 suggests some variation across tracks, but less than Soft Pop. Overall, Pop showed strong market and track-level performance when it reached a leading position, although it was not a persistent top 3 genre across all five years.

> K-pop only appeared in the top 3 once, but during that appearance it held a 10.2% market share despite having relatively low weighted track volume. It also had a high hit share of 23.2% and the highest average popularity among the top five at 77.3. Its lower standard deviation of 12 suggests relatively consistent track performance. Overall, K-pop had a strong but brief period of market leadership, with high performance despite its smaller track volume.

> Soundtrack appeared in the top 3 twice, with an average market share of 8.7%. However, it had no weighted hit share during those appearances and the lowest average popularity among the top five at 32.9. Its standard deviation of 14.3 shows moderate variation in track popularity. Overall, Soundtrack reached a leading market position more than once, but those appearances were not supported by strong track-level performance.

> Dark R&B stands out for appearing in the top 3 four times, the most among the recent results, and also has the largest weighted track volume at 91. Its average market share of 8.5% is lower than Soft Pop, Pop, and K-pop, but its frequent appearances show that it held a much more persistent market position. At the same time, its low hit share of 2.2%, lower average popularity of 49.5, and high standard deviation of 18.7 suggest weaker and less consistent track-level performance. Overall, Dark R&B shows persistent recent market leadership driven more by market presence and volume than by hit or popularity performance.

**Summary**: *Overall, the recent top 3 results show that market leadership can take different forms. Dark R&B was the most persistent leader, appearing in the top 3 in four of the five years, although its hit share and popularity were relatively weak. Pop combined strong market share with strong track performance during its two top 3 appearances, while Soft Pop also reached a strong market position twice but showed weaker and less consistent performance. K-pop stood out for strong hit share, popularity, and consistency despite reaching the top 3 only once. These results show that a genre can lead through sustained market presence, strong track performance, or a combination of both.*

### Historical Genre Performance (2000-2020)

To create a historical baseline, I looked at genre performance from 2000-2020 (*filtered out older years for relevancy*). I kept this period separate from the recent 2021-2025 data to later compare how the genre landscape has changed over time.

In [39]:
historical_df = market_share_filtered[market_share_filtered['release_year'].between(2000, 2020)]

In [40]:
# overall summary of 2000 - 2020
historical_summary = (historical_df
    .groupby('artist_genres')
    .agg(
        weighted_tracks = ('weighted_tracks', 'sum'),
        avg_market_share = ('market_share', 'mean'),
        avg_hit_share = ('hit_share', 'mean'),
        avg_popularity = ('avg_popularity', 'mean'),
        median_popularity = ('median_popularity', 'mean'),
        avg_std_popularity = ('std_popularity', 'mean'))
    .sort_values(by = 'avg_market_share', ascending = False)
)

In [41]:
historical_summary

,weighted_tracks,avg_market_share,avg_hit_share,avg_popularity,median_popularity,avg_std_popularity
artist_genres,,,,,,
soundtrack,331.833333,0.150520,0.001832,38.765587,43.428571,17.077106
pop,254.000000,0.105030,0.157580,60.413680,63.647059,18.407397
art rock,7.000000,0.074219,0.062500,66.166667,63.500000,9.192358
rap,189.333333,0.069361,0.123964,64.052972,69.944444,21.711267
soft pop,159.000000,0.066713,0.130315,62.996682,68.821429,23.238508
...,...,...,...,...,...,...
gothic rock,0.392857,0.000885,0.000000,27.000000,27.000000,NaN
southern rock,0.166667,0.000790,0.000000,8.000000,8.000000,NaN
deathrock,0.142857,0.000789,0.000000,0.000000,0.000000,NaN


> Soundtrack has the largest historical market presence, with the highest average market share at 15.1% and 331.8 weighted tracks. However, its hit share is extremely low at 0.2% and its average popularity is only 38.8. Its standard deviation of 17.1 also shows a fair amount of variation between tracks. This suggests that Soundtrack's historical presence came mainly from its large track volume rather than consistently popular or successful tracks.

> Pop has the second highest average market share at 10.5% and a large track volume of 254 weighted tracks. Unlike Soundtrack, its market presence is backed by stronger performance, with the highest hit share among the top five at 15.8% and an average popularity of 60.4. Its standard deviation of 18.4 shows that track popularity still varied quite a bit, but Pop generally combined a large market presence with strong track performance.

> Art Rock has an average market share of 7.4% and a relatively high average popularity of 66.2, with a lower standard deviation of 9.2 suggesting more consistent track performance. However, it only has 7 weighted tracks, which is much lower than the other top genres. Because of this small volume, its results should be treated more cautiously, since they may be based on a limited number of tracks rather than a broad market presence.

> Rap has a 6.9% average market share and a fairly large track volume of 189.3 weighted tracks. Its hit share of 12.4% and average popularity of 64.1 show strong track performance. However, its standard deviation of 21.7 is relatively high, meaning popularity varied quite a bit between tracks. Rap therefore had a solid historical market presence and generally strong performance, although that performance wasn't always consistent.

> Soft Pop has an average market share of 6.7% and 159 weighted tracks. Its hit share of 13% and average popularity of 63 also show relatively strong track performance. However, its standard deviation of 23.2 is the highest among the top five, meaning there was a lot of variation in how its tracks performed. Soft Pop had a solid historical presence and performed well overall, but its individual tracks were less consistent.

**Summary**: *Overall, the historical market shows that size and track performance didn't always go together. Soundtrack had by far the largest market presence but relatively weak popularity and hit performance, while Pop combined a large presence with much stronger track results. Rap and Soft Pop had smaller market shares but still performed well in popularity and hit share, although their tracks were less consistent. Art Rock also performed well despite its small volume, but its results should be interpreted more cautiously because of the limited number of tracks.*

### Historical Market Leaders

Similar to the recent analysis, I looked at genres that ranked in the top 3 by market share from 2000-2020 to show which genres historically held the strong market positions and how they performed during those years.

In [42]:
# top 3 genres in 2000 - 2020
historical_top_3 = historical_df[historical_df['genre_rank'] <= 3]

In [43]:
# summary of historical top 3
historical_top_3_summary = (historical_top_3
    .groupby('artist_genres')
    .agg(
        weighted_tracks = ('weighted_tracks', 'sum'),
        avg_market_share = ('market_share', 'mean'),
        avg_hit_share = ('hit_share', 'mean'),
        avg_popularity = ('avg_popularity', 'mean'),
        median_popularity = ('median_popularity', 'mean'),
        avg_std_popularity = ('std_popularity', 'mean'))
    .sort_values(by='avg_market_share', ascending = False)
)

In [44]:
# historical top 3 genre appearances (2000 - 2020)
historical_top_3[['artist_genres']].value_counts()

artist_genres     
soundtrack            19
pop                   10
rap                    6
soft pop               5
country                3
rock                   2
rage rap               2
edm                    2
post-grunge            2
alternative rock       2
medieval               1
r&b                    1
alternative metal      1
latin pop              1
indie                  1
hip hop                1
east coast hip hop     1
christmas              1
art rock               1
art pop                1
anime                  1
k-pop                  1
Name: count, dtype: int64

In [45]:
# adding appearance counts to the summary for more context of type of market lead
historical_top_3_summary['top_3_appearances'] = historical_top_3.groupby('artist_genres').size()

In [46]:
historical_top_3_summary

,weighted_tracks,avg_market_share,avg_hit_share,avg_popularity,median_popularity,avg_std_popularity,top_3_appearances
artist_genres,,,,,,,
hip hop,10.666667,0.172043,0.176471,58.954545,61.000000,18.274305,1
soundtrack,325.333333,0.163043,0.002024,37.488280,42.526316,18.301489,19
art pop,16.500000,0.157143,0.083333,43.687500,47.500000,25.438652,1
pop,211.500000,0.150040,0.204964,55.411958,58.650000,19.518319,10
soft pop,125.000000,0.140374,0.239741,58.681663,62.900000,21.449937,5
rap,134.333333,0.138646,0.228270,59.837290,63.083333,22.030150,6
art rock,5.500000,0.125000,0.093750,61.000000,59.000000,8.933085,1
latin pop,20.000000,0.110497,0.028571,30.523810,24.000000,16.415295,1
alternative rock,9.866667,0.110128,0.144097,55.791667,59.500000,22.541352,2


> Hip Hop has the highest average market share at 17.2%, but only appeared in the top 3 once. During that appearance, it had a strong hit share of 17.6% and an average popularity of 59. While it performed well when it reached the top 3, its single appearance suggests a brief period of strong market leadership rather than a consistent historical presence.

> Soundtrack was by far the most persistent historical market leader, appearing in the top 3 in 19 of the 21 years. It also had a high average market share of 16.3% and the largest weighted track volume at 325.3. However, its hit share was extremely low at 0.2%, with a relatively low average popularity of 37.5. This suggests that Soundtrack maintained a large and consistent market presence historically, but that presence did not translate into strong hit or popularity performance.

> Art Pop had a high average market share of 15.7%, but only appeared in the top 3 once. Its hit share of 8.3% and average popularity of 43.7 were relatively low, while its standard deviation of 25.4 was the highest among the top five. Art Pop therefore had one strong year in terms of market position, but its track performance was weaker and highly inconsistent.

> Pop shows much more sustained leadership, appearing in the top 3 in 10 of the 21 historical years. It had an average market share of 15%, a high hit share of 20.5%, and a large weighted track volume of 211.5. Its popularity varied quite a bit across tracks, but overall, Pop combined a strong market presence with strong hit performance and remained a recurring market leader throughout the historical period.

> Soft Pop appeared in the top 3 five times, with an average market share of 14.1%. It also had the highest hit share among the top five at 24% and an average popularity of 58.7. However, its relatively high standard deviation of 21.4 shows that track popularity varied considerably. Overall, Soft Pop was an occasional but successful market leader, with strong hit performance during the years it reached the top 3.

**Summary**: *Overall, historical market leadership varied a lot in both strength and persistence. Soundtrack stands out as the most consistent leader, appearing in the top 3 in 19 of 21 years, although its popularity and hit performance were relatively weak. Pop also maintained a strong presence, reaching the top 3 10 times while producing a much larger share of hits. Soft Pop and Rap appeared less frequently but showed strong track-level performance, while genres like Hip Hop and Art Pop reached high market shares during only one top 3 appearance. Overall, the results show that having a high average market share does not necessarily mean a genre consistently led the market over time.*

### Historical vs. Recent Comparison

Two main comparisons:
1. **Overall market change**: Comparing overall genre performance from 2000-2020 with 2021-2025 to identify genres that have grown, declined, or remained strong.


2. **Market leader change**: Comparing the historical and recent top 3 results to see whether the leading genres and the way they perform have changed over time.

To accomodate for the historical period covering more years than the recent period, total weighted tracks won't be compared between the two. The focus is mainly on average market share, hit share, popularity, and consistency.

### 1. Overall Market Change: 2000-2020 vs. 2021-2025
Comparing these two periods show a shift in which genres hold the largest market presence, while also showing that some genres have remained relevant over time.
* **Soundtrack**, historically had the highest average market share at 15.1%, but feel to 4.7% in the recent period. Popularity and hit share still remained relatively low in both periods, suggesting that its overall position in the market has become smaller, while holding a noticeable presence.

* **Pop** remained one of the largest genres in both periods, ranking second in average market share historically, at 10.5%, and recently 6.5%. Although market share has seemingly decreased, its average popularity increased from 60.4 to 64.5, with hit share remaining relatively strong at 13%. This suggests that Pop's overall market presence has become smaller, but its tracks continue to perform strongly.

* **Rap** also experienced a decrease in average market presence, moving from 6.9% historical average market share to 4.3% recently. Its track-level performance has improved, with average popularity increased from 64.1 to 72.6, and popularity standard deviation dropped substantially from 21.7 to 7.5. Although Rap occupies a smaller share of the recent market, its tracks appear to be performing at a higher and more consistent level.

* **Soft Pop** remained relatively similar in market presence, moving from 6.7% historical average market share to 6% recently. Its hit share decreased from 13% to 5.9%, and average popularity feel from 63 to 55.4. Its standard deviation remained high, suggesting that while Soft Pop maintained its overall market presence, its recent track performance has weakened compared to historical results.

* **Dark R&B** shows one of the clearest differences between the two periods. It did not appear among the historical top 5 by average market share, but now has the highest recent average market share at 8.1%. However, its relatively low popularity and hit share suggests that its recent market presence is currently stronger than its track-level performance.

* *Art Rock appeared among the historical top 5, but not the recent top 5. Its historical results were based on a very low track volume, so it's good to avoid treating this as its own section until further analysis.*

Overall, the recent market appears more distributed across genres, with several historically large genres holding smaller market shares. However, a decrease in market share doesn't necessarily mean weaker track performance. Market share reflects a genre's presence within the overall market, while hit share and popularity provides a better picture of how well its tracks are performing. Rap demonstrates this difference well, with a lower market share but higher and more consistent popularity. Additionally, Pop maintains strong track performance despite a smaller market share. This suggests that **market presence and track performance should be considered separately** when evaluating how a genre's position has changed over time.



### 2. Market Leader Change
*Because the historical and recent periods cover different lengths of time, raw top-3 appearance counts are not directly compared across periods. Instead, appearances are interpreted relative to the number of years in each period.*

The biggest change between the two periods is how consistently individual genres held a leading position.
* Histoically, **Soundtrack** dominated the top 3, appearing in 19 of 21 years, while **Pop** also showed a sustained leadership with 10 appearances. In the most recent period, no single genre has held the same level of dominance. **Dark R&B** came closest, appearing in 4 of 5 years, while **Pop**, **Soft Pop**, and **Soundtrack** each appeared 2 times, and **K-pop** 1 time.

* **Pop** is one of the clearest examples of continued relevance across both periods. Historically, it appeared in the top 3, 10 times with a 15% average market share and 20.5% hit share. Recently, it only appeared 2 times and its average market share fell to 12.1%, but its hit share increased to 26% and average popularity rose from 55.4 to 70.9. While Pop appears less frequenty as a market leader, its performance when it does reach the top 3 has become stronger
  
* **Soundtrack** shows the opposite pattern. It went from appearing in the historical top 3 in 19 of 21 years to only 2 times during the recent 5 year period. Its average market share during those appearances also fell from 16.3% to 8.7%, while its already weak hit and popularity performance remained low. This points to a major change in Soundtrack's position: it still occasionally reaches a leading market position, but no longer holds the persistent leadership it historically had.

* **Soft Pop** remained a recurring market leader, but its performance weakened. Its average market share during top 3 appearances decreased from 14.1% to 12.5%, while hit share dropped from 24% to 10.7% and average popularity fell from 58.7 to 43.7. This suggests that Soft Pop can still reach a strong market position, but its recent leadership is less supported by strong track performance.

The recent period also introduces different leaders.
*** Dark R&B** became the most persistent recent top 3 genre, although its leadership is driven more by market presence and track volume than strong hit performance.
*** K-pop** reached the top 3 only 1 time but showed very strong popularity and hit performance during that appearance. 
* Together, these genres show that recent leadership can come from either sustained market presence or strong performance during shorter periods of leadership.

Overall, market leadership seems to have become less concentrated in the recent period. Historically, Soundtrack and Pop repeatedly occupied leading positions, with Soundtrack appearing in the top 3 in nearly every year. Recent leadership is spread across a wider mix of genres, with Dark R&B emerging as the most persistent recent leader while Pop, Soft Pop, Soundtrack, and K-pop rotate through leading positions. The results also **reinforce** that market leadership and track performance are not the same: Dark R&B and Soundtrack achieve strong market positions despite weaker hit performance, while Pop and K-pop show much stronger track performance when they reach the top 3.

## **Part II: Genre Lifecycle**

This section compares historical and recent genre performance to see how the market has changed over time. The analysis expands beyond the top five genres to identify broader shifts, including emerging, declining, and persistent genres.